In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# CEG-WM V2：独立模型进程与本机磁盘续跑

直接 **Run all**，无需上传、手填路径或切换开关。复用原 root、首次 repair 和 compat 已完成 checkpoints，
包括已返回的 early1 VJP、action、render、clean、design_image、design_read。
仅给本次 `source06_fur_knit/bit1/early2/rgb_vjp` 一次新尝试，然后继续原固定 P2。
目标原 status 可为 STARTED/FAILED，或目录存在且累计计数证明已有消耗而 status 尚未同步；不会因此重做 early1。

每个昂贵模型操作独立子进程执行；结果落盘、子进程退出后才继续。主进程不持有模型。
保存张量按原复制语义逐张量写入 `/content/ceg_stage_protection_scratch` 本机磁盘，按需精确读取；
不向 Drive 逐张量卸载，不使用内存盘、降精度、压缩或重算。空间逐次检查；空间不足明确停止。
原种子、reader、dtype、预算、阈值与 holdout 冻结次序不变，不自动 P3/P4。

新结果写入原 attempt01 的 `repairs/source06_bit1_early2_rgb_vjp_isolated_attempt01/`。
累计从最新持久计数继续，旧失败及滞后报告差额保留。新尝试 RETURNED 复用，STARTED/FAILED 不自动循环。
额外模型加载和磁盘 I/O 会增加耗时；CPU 机制检查不代表真实 Colab/CUDA 已跑通。


In [ ]:
# 固定输出目录；再次 Run all 在同一目录恢复。
OUTPUT = '/content/drive/MyDrive/CEG-WM_Result/stage-protection-v2-20261109/attempt01'


## 准备内含源码
首格仅挂载 Drive。保持一个活动 VM；不要并行运行其他入口。
各模型 worker 的结果、退出码、计数和内存/I/O记录保存在新子目录 `workers/`，退出后清理其独占本机 scratch。


In [ ]:
import base64, gzip, io, os, sys, tarfile
from pathlib import Path
WORK = Path(os.environ.get('CEG_STAGE_PROTECTION_WORKDIR', '/content/ceg_stage_protection_v2'))
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE = '...'
with tarfile.open(fileobj=io.BytesIO(gzip.decompress(base64.b64decode(BUNDLE))), mode='r:') as archive:
    archive.extractall(WORK, filter='data')
sys.path[:0] = [str(WORK / 'src'), str(WORK)]
print('独立源码:', WORK)


## 固定方法与累计成本
原 P2 的 40 次 SD3 forward / 258 步等上限描述不间断原计划，**不再作为累计续跑成本上限**。
`continuation_counts.json` 分列原始消耗、本次新消耗、目标重试、之后续跑和累计计数。
旧中断已消耗的 attempted/returned 差额保留；不会清零或把重试当免费。

保存图不再保留整图 CPU tensor副本；仍需容纳模型、当前算子workspace和反传同时使用的张量。
磁盘峰值与I/O、进程RSS、GPU allocated/reserved分别报告，不能互相替代。


In [ ]:
import json
config = json.loads((WORK / 'configs/stage_protection_v2_colab.json').read_text(encoding='utf-8-sig'))
print(json.dumps(config, ensure_ascii=False, indent=2))
print('本次执行：固定 P2；不会自动进入 P3/P4。')

RESULTS = Path(OUTPUT) / 'repairs/source06_bit1_early2_rgb_vjp_isolated_attempt01'
print('续跑结果:', RESULTS)


## 准备环境并复用已有认证
沿用已固定的 diffusers/transformers 依赖和模型 revisions。保留 Colab 的 CUDA torch。
自动检查环境 token、已有 Hugging Face 登录缓存和 Colab secret `HF_TOKEN`。
仅在现有凭据实际无法访问固定模型时，才显示必要的权限说明与隐藏 token 输入；已有访问权限时不会强制登录。
依赖安装、模型下载、硬件资源与真实精度兼容性仍需在实际 Colab 验证。


In [ ]:
from cegwm.runtime.stage_protection_colab import ensure_model_access, record_setup_errors, run_logged
with record_setup_errors(OUTPUT):
    import torch
    assert sys.version_info >= (3, 11), '需要 Python 3.11+'
    assert tuple(int(v) for v in torch.__version__.split('+')[0].split('.')[:2]) >= (2, 6), '需要 Colab torch >= 2.6'
    assert torch.cuda.is_available(), 'Colab 当前未提供 CUDA GPU；本次未启动 P2。'
    install_code = run_logged([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                              str(WORK / 'requirements-reconstruction-channel-colab.txt')],
                             cwd=WORK, output=OUTPUT, log_name='setup.log')
    if install_code != 0:
        raise RuntimeError('依赖安装失败；完整输出已保存到 ' + str(Path(OUTPUT) / 'setup.log'))
    access = ensure_model_access(config)
    print('模型访问已确认；认证来源:', access['credential_source'])


## 运行固定的一次修复与续跑
结果直接写入上述 Drive 目录。每个操作开始前写 STARTED；发生中断不会覆盖为成功。
失败保持四试次分母；不扩来源、换 seed、换 cell、加预算或改变门槛。
runner 的 stdout/stderr 直接追加到 `runner.log`，环境准备错误保存在 `setup.log`；页面关闭不会删除已写入的日志。
`runner_resources.jsonl` 尽力记录父进程可见的 cgroup 内存事件、限制、峰值及退出差值；
`runner.log` 的 SOURCE_MEMORY 行记录长链步边界 RSS 与累计 CPU 保存字节。
缺失数据为 unknown/null，遥测写失败不阻止运行；cgroup 是共享范围，不能单凭 -9 或计数变化认定本进程 OOM。

`WORKER_MEMORY` 观察 SDXL encode、每步、decode、backward 前后；worker receipt保留exitcode及最新计数，SIGKILL后也会回收scratch。


In [ ]:
environment = os.environ.copy()
environment['PYTHONPATH'] = os.pathsep.join([str(WORK / 'src'), str(WORK)])
returncode = run_logged([sys.executable, '-u', '-m', 'experiments.run_stage_protection_v2_isolated_resume',
                         '--execute', '--output', OUTPUT], cwd=WORK, env=environment, output=OUTPUT)
print('进程退出码:', returncode, '持久日志:', str(Path(OUTPUT) / 'runner.log'))
if returncode not in (0, 2):
    print('进程未正常完成。下方如有结果，是目录中最后保存的结果；请结合 runner.log 查看本次错误。')


## 查看全部试次、原始 proposal 与共同回退
共同三通道指标是 clean/design 固定 cell 任务及 holdout 盲读目标 bit；预算另列。
另列 holdout 相对 design 固定 cell 的位置诊断，避免把不同同 bit cell 混为固定 cell 成功。
准出按内容成组：两个内容均需主动 early 翻转、A 干扰、非零 C 保护和严格 C>B 改进，
另一 bit 不得出现 B 成功 C 失败。质量只报告；有明显质量代价时需审查取舍，不能宣称已隔离机制收益。
即使出现操作性信号，P3 也需来源审计会话独立审阅，不自动扩跑。


In [ ]:
if (RESULTS / 'summary.json').exists():
    import pandas as pd
    from IPython.display import display
    summary = json.loads((RESULTS / 'summary.json').read_text())
    rows = []
    for trial in summary['trials']:
        for arm in ('A', 'B', 'C'):
            proposal = trial.get('arms', {}).get(arm, {})
            rows.append({'source': trial['source_id'], 'bit': trial['bit'], 'arm': arm,
                'status': proposal.get('status', 'incomplete'), 'selected': proposal.get('selected'),
                'raw_success': proposal.get('raw_three_channel_success'),
                'returned_success': proposal.get('returned_three_channel_success'),
                'protection_realized': proposal.get('protection_realized') if arm == 'C' else None,
                'C_B_actual_different': trial.get('C_B_actual_different') if arm == 'C' else None,
                'protection_active': trial.get('protection_constraint_active') if arm == 'C' else None,
                'early_relative': trial.get('early_relative'), 'late_relative': proposal.get('late_relative'),
                'RGB_MSE': proposal.get('quality', {}).get('mse'),
                'design_margin_change': proposal.get('design_margin_change'),
                'errors': '; '.join(trial.get('errors', []))})
    display(pd.DataFrame(rows))
    print(json.dumps(summary['verdict'], ensure_ascii=False, indent=2))
    display(pd.DataFrame([summary['physical_counts']]).T.rename(columns={0: 'observed_count'}))
    print('工程完成:', summary['engineering_complete'], '完整记录:', OUTPUT)
else:
    print('尚无 P2 汇总；检查 runner.log、setup.log 与已保存的操作记录，不代表实验通过。')

if (RESULTS / 'continuation_counts.json').exists():
    print('最近一次派生计数报告（SIGKILL 时可能滞后）：')
    print(json.dumps(json.loads((RESULTS / 'continuation_counts.json').read_text()), ensure_ascii=False, indent=2))
for label, path in (
    ('当前持久累计计数', RESULTS / 'physical_counts.json'),
    ('当前一次重试状态', RESULTS / 'operations/source06_fur_knit/bit1/early2/rgb_vjp/status.json'),
):
    if path.exists():
        print(label, json.dumps(json.loads(path.read_text()), ensure_ascii=False, indent=2))


In [ ]:
if (RESULTS / 'evaluated.pt').exists():
    import torch
    import matplotlib.pyplot as plt
    evaluated = torch.load(RESULTS / 'evaluated.pt', map_location='cpu', weights_only=False)
    for trial in evaluated['trials']:
        if 'E' not in trial:
            print(trial['source_id'], trial['bit'], trial.get('errors'))
            continue
        panels = [('E', trial['E']['image'])]
        for arm in ('A', 'B', 'C'):
            proposal = trial['arms'].get(arm, {})
            observation = proposal.get('observation') or proposal.get('render')
            if observation:
                panels.append((arm + ' raw; return=' + proposal['selected'], observation['image']))
        fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4), squeeze=False)
        for ax, (label, image) in zip(axes[0], panels):
            ax.imshow(image); ax.set_title(label); ax.axis('off')
        fig.suptitle(trial['source_id'] + ' / bit' + str(trial['bit']))
        plt.tight_layout(); plt.show()
